# Evaluation Script: Systematic Policy Weakening (Mixed Content)

This script produces results for Section 5.6 Sytematic Policy Weakening

### Connect to the MongoDB

In [ ]:
import tqdm
import pymongo

client = pymongo.MongoClient("mongodb://localhost:27017/")

db = client["webview"]
library_collection = db["library"]
network_logs_collection = db["network_logs"]

dynamic_api_calls_collection = db["dynamic_api_calls"]

def print_latex_macro(name: str, value: str):
    print(f"\\newcommand{{\\{name}}}{{{value}}}")

### Retrieve stats on the usage of `setMixedContentMode`

Produces results for "Prevalence"

In [2]:
total_apps_dynamic_calls = dynamic_api_calls_collection.distinct("source_package_name", {})

sets_mc_mode = dynamic_api_calls_collection.distinct("source_package_name", {"api": "WEB_SETTINGS_SET_MIXED_CONTENT_MODE"})
print_latex_macro("dynamicAppsSettingMixedContentMode", f"{len(list(sets_mc_mode)):,}")
print_latex_macro("dynamicAppsSettingMixedContentModePercent", f"{(len(list(sets_mc_mode)) / len(list(total_apps_dynamic_calls))) * 100:.2f}")

# get how many set it to 0 or 2
sets_0_or_two = dynamic_api_calls_collection.distinct("source_package_name", {"api": "WEB_SETTINGS_SET_MIXED_CONTENT_MODE", "params.0": {"$in": [0, 2]}})
sets_0 = dynamic_api_calls_collection.distinct("source_package_name", {"api": "WEB_SETTINGS_SET_MIXED_CONTENT_MODE", "params.0": 0})
sets_2 = dynamic_api_calls_collection.distinct("source_package_name", {"api": "WEB_SETTINGS_SET_MIXED_CONTENT_MODE", "params.0": 2})
print_latex_macro("dynamicAppsSettingMixedContentModeZeroOrTwo", f"{len(list(sets_0_or_two)):,}")
print_latex_macro("dynamicAppsSettingMixedContentModeZeroOrTwoPercent", f"{(len(list(sets_0_or_two)) / len(list(total_apps_dynamic_calls))) * 100:.2f}")
print_latex_macro("dynamicAppsSettingMixedContentModeZero", f"{len(list(sets_0)):,}")
print_latex_macro("dynamicAppsSettingMixedContentModeZeroPercent", f"{(len(list(sets_0)) / len(list(total_apps_dynamic_calls))) * 100:.2f}")
print_latex_macro("dynamicAppsSettingMixedContentModeTwo", f"{len(list(sets_2)):,}")
print_latex_macro("dynamicAppsSettingMixedContentModeTwoPercent", f"{(len(list(sets_2)) / len(list(total_apps_dynamic_calls))) * 100:.2f}")

\newcommand{\dynamicAppsSettingMixedContentMode}{18,204}
\newcommand{\dynamicAppsSettingMixedContentModePercent}{72.28}
\newcommand{\dynamicAppsSettingMixedContentModeZeroOrTwo}{17,620}
\newcommand{\dynamicAppsSettingMixedContentModeZeroOrTwoPercent}{69.96}
\newcommand{\dynamicAppsSettingMixedContentModeZero}{3,312}
\newcommand{\dynamicAppsSettingMixedContentModeZeroPercent}{13.15}
\newcommand{\dynamicAppsSettingMixedContentModeTwo}{15,578}
\newcommand{\dynamicAppsSettingMixedContentModeTwoPercent}{61.85}


### Retrieve the libraries where `setMixedContentMode` calls originate from

In [12]:
lib_cache = {}

results_size = dynamic_api_calls_collection.count_documents({"api": "WEB_SETTINGS_SET_MIXED_CONTENT_MODE", "params.0": {"$in": [0, 2]}})
results = dynamic_api_calls_collection.find({"api": "WEB_SETTINGS_SET_MIXED_CONTENT_MODE", "params.0": {"$in": [0, 2]}})

In [13]:
dynamic_params_0 = []
dynamic_params_2 = []

def get_library_for_class_package(class_name):
    # get all library identifiers 
    if "library_identifiers" in lib_cache:
        library_identifiers = lib_cache["library_identifiers"]
    else:
        library_identifiers = list(library_collection.find({}, {"identifier": 1, "_id": 0}))
        lib_cache["library_identifiers"] = library_identifiers
    
    matching_identifiers = [lib["identifier"] for lib in library_identifiers if class_name.startswith(lib["identifier"] + ".")]
    if not matching_identifiers:
        return None
        
    # get the longest matching identifier
    matching_identifiers = sorted(matching_identifiers, key=lambda x: len(x), reverse=True)
    longest_matching_identifier = matching_identifiers[0]
    # query the library mapping collection for the longest matching identifier and the package name
    return longest_matching_identifier

for r in tqdm.tqdm(results, total=results_size):
    package_name = r["source_package_name"]
    mode = r["params"][0]
    caller_clazz = r.get("trace", [])[0]
    library = get_library_for_class_package(caller_clazz)
    if caller_clazz and caller_clazz.lower().startswith(package_name.lower()):
        library = "first-party"
    if mode == 2:
        dynamic_params_2.append((package_name, caller_clazz, library))
    elif mode == 0:
        dynamic_params_0.append((package_name, caller_clazz, library))

100%|██████████| 410126/410126 [02:51<00:00, 2388.51it/s]


### Generate the top 10 sources allowing mixed content table

Produces Table 3 (Section 5.6)

In [21]:
import numpy as np
import pandas as pd
# put dynamic_params_0 into a dataframe
df_dynamic_0 = pd.DataFrame(dynamic_params_0, columns=['package_name', 'class', 'library'])
df_dynamic_2 = pd.DataFrame(dynamic_params_2, columns=['package_name', 'class', 'library'])

df_dynamic_0['library'] = df_dynamic_0['library'].fillna("Source Unknown")
df_dynamic_2['library'] = df_dynamic_2['library'].fillna("Source Unknown")
# show all where library is "Unknown"

# for each library, count how many unique package names set mixed content mode to 0
library_counts_0 = df_dynamic_0.groupby('library')['package_name'].nunique().reset_index()
library_counts_0 = library_counts_0.sort_values(by='package_name', ascending=False)

library_counts_2 = df_dynamic_2.groupby('library')['package_name'].nunique().reset_index()
library_counts_2 = library_counts_2.sort_values(by='package_name', ascending=False)

# combine library_counts_0 and library_counts_2 into a single dataframe
library_counts_combined = pd.merge(library_counts_0, library_counts_2, on='library', how='outer', suffixes=('_0', '_2')).fillna(0)
library_counts_combined["higher_value"] = library_counts_combined[['package_name_0', 'package_name_2']].max(axis=1)
# sort by higher_value descending
library_counts_combined = library_counts_combined.sort_values(by=['higher_value'], ascending=False)
library_counts_combined['package_name_0_percent'] = library_counts_combined['package_name_0'] / len(total_apps_dynamic_calls) * 100
library_counts_combined['package_name_2_percent'] = library_counts_combined['package_name_2'] / len(total_apps_dynamic_calls) * 100


def get_library_category_name(library_identifier: str) -> tuple | None:
    if library_identifier == "1st-party":
        return "1st-party", "1st-party"
    info = library_collection.find_one({"identifier": library_identifier}, {"category": 1, "name": 1, "_id": 0})
    category = info["category"] if info else "Unknown-Source"
    name = info["name"] if info else "Unknown Source"
    return category, name

table = """
\\begin{table}[]
    \\centering
    \\caption{Top 10 libraries dynamically allowing mixed content}
    \\label{tab:dynamic-mixed-content-libraries}
    \\footnotesize
    \\begin{tabular}{l l l l}
        \\toprule
        \\textbf{Library} & \\textbf{Category} & \\textbf{\\# Active Apps} & \\textbf{\\# Passive Apps} \\\\
        \\midrule
"""
for index, row in library_counts_combined.head(15).iterrows():
    library = row['library'] if row['library'] is not None else "Unknown"
    if library == "first-party":
        category = "first-party"
        name = "first-party"
    else:
        category, name = get_library_category_name(library)
    if library == "Source Unknown":
        continue
    active_apps = int(row['package_name_0'])
    active_apps_percent = row['package_name_0_percent']
    passive_apps = int(row['package_name_2'])
    passive_apps_percent = row['package_name_2_percent']
    table += f"        {name} & {category} & {active_apps:,} ({active_apps_percent:.2f}\\%) & {passive_apps:,} ({passive_apps_percent:.2f}\\%) \\\\\n"
table += """        \\bottomrule
    \\end{tabular}
\\end{table}
"""

print(table)


\begin{table}[]
    \centering
    \caption{Top 10 libraries dynamically allowing mixed content}
    \label{tab:dynamic-mixed-content-libraries}
    \footnotesize
    \begin{tabular}{l l l l}
        \toprule
        \textbf{Library} & \textbf{Category} & \textbf{\# Active Apps} & \textbf{\# Passive Apps} \\
        \midrule
        Google Mobile Ads & Advertising & 0 (0.00\%) & 15,164 (60.21\%) \\
        first-party & first-party & 609 (2.42\%) & 85 (0.34\%) \\
        Fyber SDK & Advertising & 0 (0.00\%) & 564 (2.24\%) \\
        Mintegral & Advertising & 549 (2.18\%) & 0 (0.00\%) \\
        Classplus & Unknown & 533 (2.12\%) & 0 (0.00\%) \\
        Ionic Cordova WebView & Hybrid Framework & 203 (0.81\%) & 0 (0.00\%) \\
        Google Mobile Services & Platform & 82 (0.33\%) & 4 (0.02\%) \\
        just & UI Components & 68 (0.27\%) & 0 (0.00\%) \\
        im.delight.android.webview & UI Components & 67 (0.27\%) & 19 (0.08\%) \\
        Flutter InAppWebView & UI Components & 21 (0.

### Retrieve the amount of apps where mixed content is transmitted

Produces results for "Recorded Mixed Content"

In [6]:
results = network_logs_collection.find({"initiator": {"$regex": "^https"}, "request_type": {"$ne": "main frame"}}).distinct("package_name")
print_latex_macro("appsMixedContent", f"{len((results)):,}")

\newcommand{\appsMixedContent}{182}
